# Software Engineering Practices

---

## 1. Code Quality

### Code Review
A systematic examination of source code by peers to catch bugs, improve readability, and share knowledge.

**Benefits:** catches defects early, enforces standards, spreads knowledge.  
**Drawbacks:** time-consuming, can create bottlenecks, may become rubber-stamping if not taken seriously.

```
# Typical PR checklist
- [ ] Logic is correct and edge cases handled
- [ ] Naming is clear and consistent
- [ ] No duplicated code (DRY)
- [ ] Tests are present and passing
- [ ] No security issues (hardcoded secrets, SQL injection vectors)
```

### Static Code Analysis
Analyzes source code **without executing it**. Tools: ESLint, SonarQube, Checkstyle, Pylint.

**Benefits:** fast feedback, integrates into CI, finds common bugs automatically.  
**Drawbacks:** false positives, doesn't catch runtime errors.

```bash
# Example: run ESLint on a JS project
npx eslint src/ --ext .js,.ts
```

### Dynamic Code Analysis
Analyzes code **during execution** (profiling, memory leak detection). Tools: Valgrind, JaCoCo, Dynatrace.

**Benefits:** finds real runtime issues (memory leaks, race conditions).  
**Drawbacks:** requires running the app, coverage depends on test scenarios.

---

## 2. CI/CD

**Continuous Integration (CI):** Developers merge code frequently; each merge triggers automated build + tests.  
**Continuous Delivery (CD):** Every passing build is automatically deployable to a staging/prod environment.  
**Continuous Deployment:** Extends CD — every passing build is *automatically released* to production.

### Key Principles
- Fail fast: catch issues as early as possible.
- Single source of truth: the pipeline is the gate to production.
- Reproducible builds: same input → same output, always.

### Simple Pipeline Example (GitHub Actions)

```yaml
name: CI
on: [push, pull_request]
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install dependencies
        run: npm install
      - name: Lint
        run: npm run lint
      - name: Test
        run: npm test
      - name: Build
        run: npm run build
```

### Analysing Bottlenecks
Common pipeline bottlenecks:
- Long-running test suites → parallelize or split test tiers.
- Heavy Docker builds → use layer caching.
- Sequential stages that could run in parallel → use DAG-style pipelines.

---

## 3. Version Control Systems (VCS)

### Centralized vs. Distributed

| Feature | Centralized (SVN, TFVC) | Distributed (Git, Mercurial) |
|---|---|---|
| Single server | Yes | No — every clone is a full repo |
| Offline work | Limited | Full — commit, branch, log locally |
| Performance | Network-dependent | Fast local operations |
| Branching | Heavyweight | Lightweight |

### Core Git Concepts

```bash
# Stage and commit
git add .
git commit -m "feat: add login endpoint"

# Branching
git checkout -b feature/my-feature
git merge feature/my-feature          # merge into current branch
git rebase main                       # replay commits on top of main

# Stashing (save work-in-progress temporarily)
git stash
git stash pop

# Forking: create your own server-side copy of a repo (common in open-source)
# Pull/Merge Request: propose changes from your branch/fork → target branch

# Advanced
git tag v1.0.0                        # mark a release
git hook (pre-commit, pre-push)       # scripts that run on git events
git remote prune origin               # remove stale remote-tracking branches
git gc --prune=now                    # clean up unreachable objects locally
```

---

## 4. Branching Strategies

### GitFlow
Structured model with long-lived branches: `main`, `develop`, `feature/*`, `release/*`, `hotfix/*`.

```
main ─────────────────────────────────── (production)
       └─ develop ──────────────────────
              └─ feature/login ─── merged back to develop
```

**Best for:** products with scheduled releases, teams needing clear release management.  
**Drawback:** complex, merges can be painful.

### GitHub Flow
Simpler: one `main` branch + short-lived feature branches. Merge via Pull Request.

```
main ───────────────────────────────────
        └─ feature/x ── PR review ── merge
```

**Best for:** teams deploying continuously, small-to-medium projects.

### Trunk-Based Development
Everyone commits to `main` (or very short-lived branches). Feature flags control incomplete work.  
**Best for:** high-velocity teams with strong CI/CD.

---

## 5. Testing Fundamentals & Testing Pyramid

```
        /‾‾‾‾‾‾‾‾‾‾\
       /  E2E Tests  \       ← Slow, expensive, few
      /──────────────\
     / Integration    \      ← Medium speed/cost
    /──────────────────\
   /    Unit Tests      \    ← Fast, cheap, many
  /──────────────────────\
```

### Unit Tests
Test a single function or class in isolation. Dependencies are mocked.

```python
# Python example
def add(a, b):
    return a + b

def test_add():
    assert add(2, 3) == 5   # fast, no I/O
```

**Pros:** fast, pinpoint failures.  **Cons:** don't test integration.

### Integration Tests
Test that components work together (e.g., service + database).

```python
# pytest + SQLAlchemy example
def test_should_save_and_retrieve_user(db_session, user_service, user_repository):
    user_service.save(User(name="Alice"))
    assert user_repository.find_by_name("Alice") is not None
```

**Pros:** catch integration bugs.  **Cons:** slower, need running infrastructure.

### End-to-End (E2E) Tests
Simulate real user flows through the full stack.

```python
# Playwright (Python) example
def test_user_can_log_in(page):
    page.goto("/login")
    page.fill("#email", "user@test.com")
    page.fill("#password", "secret")
    page.click("button[type=submit]")
    expect(page).to_have_url("/dashboard")
```

**Pros:** highest confidence.  **Cons:** slow, flaky, expensive to maintain.

---

## 6. Non-Functional Requirements & Performance Testing

### Key NFRs (Performance-Related)
| NFR | Definition | How Measured |
|---|---|---|
| Response Time | Time to process a request | p50, p90, p99 latency (ms) |
| Throughput | Requests handled per second | RPS / TPS |
| Availability | % uptime | (uptime / total time) × 100 |
| Scalability | Behavior under growing load | Load tests at N×, 10N× users |
| Resource Usage | CPU, memory, disk, network | Monitoring dashboards |

### Types of Performance Testing

- **Load Testing:** Simulate expected user load to validate normal behaviour.
- **Stress Testing:** Push beyond normal limits to find the breaking point.
- **Spike Testing:** Sudden surge in users (e.g., flash sale).
- **Soak/Endurance Testing:** Sustained load over a long period to detect memory leaks.
- **Scalability Testing:** Gradually increase load to see how the system scales.

```python
# locust load test example
from locust import HttpUser, task, between

class ProductUser(HttpUser):
    wait_time = between(1, 1)  # 1 second wait between tasks

    @task
    def get_products(self):
        self.client.get("/products")

# Run with: locust --host=https://api.example.com --users=100 --spawn-rate=10
```

---

## 7. Establishing & Improving Practices

### Identifying Gaps
Questions to ask:
- Is there a documented branching strategy? Is it followed?
- Are code reviews happening consistently, or are PRs rubber-stamped?
- Does the CI pipeline run on every PR, or only on main?
- Is test coverage tracked? Are there critical paths with zero tests?
- Are performance SLAs defined and tested against?

### Proposing Improvements
- Prioritize by **risk and impact**: untested critical paths first.
- Automate quality gates in CI (linting, coverage thresholds, SAST scans).
- Use **Definition of Done** to encode quality standards into the team process.
- Run retrospectives focused on engineering practice health.
- Organize **knowledge-sharing sessions** (e.g., internal talks, code katas, brown bags) to spread expertise across the team.

---

> **Summary:** Good engineering practices compound over time. A small investment in CI/CD, consistent testing, clean code reviews, and thoughtful branching dramatically reduces defect rates, deployment risk, and onboarding friction.